# HW14: Embeddings, FAISS, Retrieval Evaluation and Mini-RAG

В этой работе собран небольшой воспроизводимый retrieval pipeline:

- локальная база знаний по темам retrieval, ML workflow и mini-RAG;
- чанкинг документов с overlap;
- TF-IDF в качестве лёгкого способа векторизации;
- `FAISS` индекс для top-k поиска;
- оценка retrieval через `hit@k`, `recall@k`, `MRR@k`;
- сравнение двух значений `chunk_size`;
- обновление базы знаний и переиндексация;
- простой mini-RAG с ответом и источниками.

## План ноутбука

1. Импорты, seed и среда.
2. Загрузка базы знаний и sanity-check.
3. Чанкинг документов и примеры чанков.
4. Построение векторов и индекса `FAISS`.
5. Контрольные запросы и оценка retrieval.
6. Небольшой эксперимент по параметрам.
7. Обновление базы знаний и сравнение до/после.
8. Mini-RAG и короткий анализ ошибок.
9. Генерация `report.md`.

In [ ]:
import importlib.util
import json
import random
import re
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

if importlib.util.find_spec("faiss") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "faiss-cpu"])

import faiss

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_columns", 20)

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

CANDIDATE_ROOTS = [
    Path.cwd(),
    Path.cwd() / "homeworks" / "HW14",
    Path.cwd().parent,
    Path.cwd().parent / "homeworks" / "HW14",
]

ROOT_DIR = None
for candidate in CANDIDATE_ROOTS:
    if (candidate / "knowledge_base.json").exists() and (candidate / "S14-homework.md").exists():
        ROOT_DIR = candidate.resolve()
        break

if ROOT_DIR is None:
    raise FileNotFoundError("Could not locate HW14 root directory with knowledge_base.json")

ARTIFACTS_DIR = ROOT_DIR / "artifacts"
FIGURES_DIR = ARTIFACTS_DIR / "figures"
KB_PATH = ROOT_DIR / "knowledge_base.json"
KB_UPDATE_PATH = ROOT_DIR / "knowledge_base_update.json"
REPORT_TEMPLATE_PATH = ROOT_DIR / "S14-hw-report-template.md"
REPORT_PATH = ROOT_DIR / "report.md"

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

TOP_K = 3
MAIN_CHUNK_SIZE = 320
MAIN_OVERLAP = 70

print("ROOT_DIR:", ROOT_DIR)
print("Python:", sys.version.split()[0])
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("faiss:", getattr(faiss, "__version__", "n/a"))

## 1. База знаний и первичный анализ

В качестве базы знаний используется локально собранный набор коротких документов по теме retrieval, evaluation и ML workflow.
Такая область удобна для учебного mini-RAG: документы тематически связаны, но содержат разные аспекты одной темы.

In [ ]:
kb_docs = json.loads(KB_PATH.read_text(encoding="utf-8"))
kb_df = pd.DataFrame(kb_docs)
kb_df["char_count"] = kb_df["text"].str.len()
kb_df["word_count"] = kb_df["text"].str.split().str.len()

print("Number of documents:", len(kb_df))
kb_df[["source", "title", "char_count", "word_count"]]

In [ ]:
sample_columns = ["source", "title", "text"]
kb_df[sample_columns].head(4)

## 2. Чанкинг документов

Используем простой и воспроизводимый способ: фиксированные окна по символам с overlap.
Такой подход достаточно прозрачен для учебного retrieval-эксперимента.

In [ ]:
def normalize_text(text: str) -> str:
    text = text.replace("\r", " ").replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def chunk_text(text: str, chunk_size: int = 320, overlap: int = 70):
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")
    clean_text = normalize_text(text)
    if len(clean_text) <= chunk_size:
        return [(0, len(clean_text), clean_text)]

    chunks = []
    start = 0
    step = chunk_size - overlap
    while start < len(clean_text):
        end = min(len(clean_text), start + chunk_size)
        chunk = clean_text[start:end].strip()
        if chunk:
            chunks.append((start, end, chunk))
        if end == len(clean_text):
            break
        start += step
    return chunks


def build_chunks(documents_df: pd.DataFrame, chunk_size: int, overlap: int) -> pd.DataFrame:
    rows = []
    for doc in documents_df.to_dict("records"):
        windows = chunk_text(doc["text"], chunk_size=chunk_size, overlap=overlap)
        for chunk_idx, (start, end, chunk) in enumerate(windows):
            rows.append(
                {
                    "source": doc["source"],
                    "title": doc["title"],
                    "chunk_id": f'{doc["source"]}_chunk_{chunk_idx:02d}',
                    "chunk_index": chunk_idx,
                    "start_char": start,
                    "end_char": end,
                    "chunk_text": chunk,
                    "vector_text": f'{doc["title"]}. {chunk}',
                }
            )
    return pd.DataFrame(rows)


chunks_df = build_chunks(kb_df, chunk_size=MAIN_CHUNK_SIZE, overlap=MAIN_OVERLAP)
print("Number of chunks:", len(chunks_df))
chunks_df.head(8)

In [ ]:
example_source = kb_df.iloc[0]["source"]
example_doc = kb_df.loc[kb_df["source"] == example_source, ["source", "title", "text"]].iloc[0]
example_chunks = chunks_df.loc[chunks_df["source"] == example_source, ["chunk_id", "start_char", "end_char", "chunk_text"]]

print("Document example:")
print(example_doc["title"])
print(example_doc["text"][:500] + "...")
print()
print("Chunks from the same document:")
example_chunks

In [ ]:
chunk_examples_path = ARTIFACTS_DIR / "chunk_examples.csv"
chunks_df[["source", "chunk_id", "start_char", "end_char", "chunk_text"]].to_csv(chunk_examples_path, index=False)
chunk_examples_path

## 3. Векторизация и индекс FAISS

Для компактного и воспроизводимого решения используем `TfidfVectorizer`.
Это не нейросетевая embedding-модель, но это корректный и прозрачный способ построить векторные представления чанков для учебного retrieval.

In [ ]:
def l2_normalize(matrix: np.ndarray) -> np.ndarray:
    matrix = np.asarray(matrix, dtype=np.float32)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms = np.clip(norms, 1e-12, None)
    return matrix / norms


def build_retrieval_system(chunks: pd.DataFrame):
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
    chunk_matrix = vectorizer.fit_transform(chunks["vector_text"])
    chunk_vectors = l2_normalize(chunk_matrix.toarray())

    index = faiss.IndexFlatIP(chunk_vectors.shape[1])
    index.add(chunk_vectors.astype(np.float32))

    return {
        "chunks": chunks.reset_index(drop=True).copy(),
        "vectorizer": vectorizer,
        "vectors": chunk_vectors,
        "index": index,
    }


def retrieve(query: str, system: dict, top_k: int = 3) -> pd.DataFrame:
    query_vector = system["vectorizer"].transform([query]).toarray()
    query_vector = l2_normalize(query_vector).astype(np.float32)
    scores, indices = system["index"].search(query_vector, top_k)

    rows = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        chunk_row = system["chunks"].iloc[int(idx)]
        rows.append(
            {
                "rank": rank,
                "score": float(score),
                "source": chunk_row["source"],
                "title": chunk_row["title"],
                "chunk_id": chunk_row["chunk_id"],
                "chunk_text": chunk_row["chunk_text"],
            }
        )
    return pd.DataFrame(rows)


retrieval_system = build_retrieval_system(chunks_df)
print("Chunk vector shape:", retrieval_system["vectors"].shape)
print("FAISS index size:", retrieval_system["index"].ntotal)

In [ ]:
sample_queries = [
    "Why should the test split be used only once?",
    "How can TF-IDF vectors be indexed with FAISS?",
    "Why is overlap helpful during chunking?",
    "Why should a mini RAG return sources?",
    "What does error analysis help separate?",
]

for query in sample_queries:
    print("=" * 100)
    print("Query:", query)
    display(retrieve(query, retrieval_system, top_k=TOP_K)[["rank", "score", "source", "chunk_id"]])

## 4. Контрольные запросы и оценка retrieval

Ниже зафиксирован небольшой набор контрольных запросов.
Для части запросов допускается несколько релевантных источников, поэтому `recall@k` несёт чуть больше информации, чем просто `hit@k`.

In [ ]:
eval_queries = [
    {
        "query": "Why should the test split be used only once?",
        "expected_sources": ["doc_train_val_test", "doc_data_leakage"],
    },
    {
        "query": "Why is random shuffling wrong for time series evaluation?",
        "expected_sources": ["doc_train_val_test", "doc_time_series_split"],
    },
    {
        "query": "What does overlap protect against in chunking?",
        "expected_sources": ["doc_chunking_strategies"],
    },
    {
        "query": "How can TF-IDF vectors be used together with FAISS?",
        "expected_sources": ["doc_tfidf_vectorizer", "doc_faiss_index"],
    },
    {
        "query": "What is hit@k and when can it match recall@k?",
        "expected_sources": ["doc_retrieval_metrics"],
    },
    {
        "query": "Why should a mini RAG return sources with the answer?",
        "expected_sources": ["doc_rag_pipeline", "doc_prompt_answering"],
    },
    {
        "query": "What kinds of mistakes should error analysis separate?",
        "expected_sources": ["doc_error_analysis"],
    },
    {
        "query": "Why is experiment tracking useful in retrieval experiments?",
        "expected_sources": ["doc_model_selection"],
    },
    {
        "query": "Which similarity setup is common with IndexFlatIP?",
        "expected_sources": ["doc_faiss_index", "doc_embeddings_basics"],
    },
    {
        "query": "What are the limitations of template based answering?",
        "expected_sources": ["doc_prompt_answering"],
    },
]

eval_df = pd.DataFrame(eval_queries)
eval_df["expected_source"] = eval_df["expected_sources"].apply(lambda items: " | ".join(items))
eval_df[["query", "expected_source"]]

In [ ]:
def unique_in_order(items):
    result = []
    seen = set()
    for item in items:
        if item not in seen:
            result.append(item)
            seen.add(item)
    return result


def evaluate_retrieval(eval_rows, system, top_k: int = 3) -> pd.DataFrame:
    records = []
    for row in eval_rows:
        retrieved = retrieve(row["query"], system, top_k=top_k)
        retrieved_sources = unique_in_order(retrieved["source"].tolist())
        expected = row["expected_sources"]
        hits = [source for source in expected if source in retrieved_sources]
        hit_at_k = int(len(hits) > 0)
        recall_at_k = len(hits) / len(expected)

        rank_of_first_relevant = None
        reciprocal_rank = 0.0
        for rank, source in enumerate(retrieved["source"].tolist(), start=1):
            if source in expected:
                rank_of_first_relevant = rank
                reciprocal_rank = 1.0 / rank
                break

        records.append(
            {
                "query": row["query"],
                "expected_source": " | ".join(expected),
                "retrieved_sources": " | ".join(retrieved_sources),
                "hit_at_k": hit_at_k,
                "recall_at_k": recall_at_k,
                "rank_of_first_relevant": rank_of_first_relevant,
                "mrr_component": reciprocal_rank,
            }
        )
    return pd.DataFrame(records)


retrieval_eval_df = evaluate_retrieval(eval_queries, retrieval_system, top_k=TOP_K)
retrieval_eval_path = ARTIFACTS_DIR / "retrieval_eval.csv"
retrieval_eval_df.to_csv(retrieval_eval_path, index=False)

retrieval_metrics = {
    "top_k": TOP_K,
    "hit_at_k": float(retrieval_eval_df["hit_at_k"].mean()),
    "recall_at_k": float(retrieval_eval_df["recall_at_k"].mean()),
    "mrr_at_k": float(retrieval_eval_df["mrr_component"].mean()),
    "num_queries": int(len(retrieval_eval_df)),
}

metrics_summary_path = ARTIFACTS_DIR / "retrieval_metrics_summary.json"
metrics_summary_path.write_text(json.dumps(retrieval_metrics, indent=2), encoding="utf-8")

retrieval_eval_df

In [ ]:
metrics_table = pd.DataFrame(
    [
        {"metric": "hit@3", "value": retrieval_metrics["hit_at_k"]},
        {"metric": "recall@3", "value": retrieval_metrics["recall_at_k"]},
        {"metric": "MRR@3", "value": retrieval_metrics["mrr_at_k"]},
    ]
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(metrics_table["metric"], metrics_table["value"], color=["#4C72B0", "#55A868", "#C44E52"])
ax.set_ylim(0, 1.05)
ax.set_title("Retrieval quality on control queries")
for idx, value in enumerate(metrics_table["value"]):
    ax.text(idx, value + 0.03, f"{value:.2f}", ha="center", fontsize=10)
plt.tight_layout()
retrieval_plot_path = FIGURES_DIR / "retrieval_quality_plot.png"
plt.savefig(retrieval_plot_path, dpi=160, bbox_inches="tight")
plt.show()

metrics_table

## 5. Небольшой эксперимент по параметрам retrieval

Сравним два значения `chunk_size` при фиксированном overlap и посмотрим, как меняется retrieval quality.

In [ ]:
experiment_rows = []
for candidate_chunk_size in [220, 320]:
    candidate_chunks = build_chunks(kb_df, chunk_size=candidate_chunk_size, overlap=MAIN_OVERLAP)
    candidate_system = build_retrieval_system(candidate_chunks)
    candidate_eval = evaluate_retrieval(eval_queries, candidate_system, top_k=TOP_K)

    experiment_rows.append(
        {
            "chunk_size": candidate_chunk_size,
            "overlap": MAIN_OVERLAP,
            "num_chunks": len(candidate_chunks),
            "hit_at_k": candidate_eval["hit_at_k"].mean(),
            "recall_at_k": candidate_eval["recall_at_k"].mean(),
            "mrr_at_k": candidate_eval["mrr_component"].mean(),
        }
    )

experiment_df = pd.DataFrame(experiment_rows).sort_values("chunk_size").reset_index(drop=True)
experiment_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(experiment_df["chunk_size"], experiment_df["hit_at_k"], marker="o", label="hit@3")
axes[0].plot(experiment_df["chunk_size"], experiment_df["recall_at_k"], marker="o", label="recall@3")
axes[0].set_title("Metrics vs chunk_size")
axes[0].set_xlabel("chunk_size")
axes[0].set_ylim(0, 1.05)
axes[0].legend()

axes[1].bar(experiment_df["chunk_size"].astype(str), experiment_df["num_chunks"], color="#8172B3")
axes[1].set_title("Number of chunks")
axes[1].set_xlabel("chunk_size")

plt.tight_layout()
experiment_plot_path = FIGURES_DIR / "chunk_size_experiment.png"
plt.savefig(experiment_plot_path, dpi=160, bbox_inches="tight")
plt.show()

## 6. Обновление базы знаний и переиндексация

Добавим несколько новых документов, заново выполним чанкинг и построение индекса, а затем сравним retrieval до и после обновления.

In [ ]:
kb_update_docs = json.loads(KB_UPDATE_PATH.read_text(encoding="utf-8"))
kb_update_df = pd.DataFrame(kb_update_docs)
kb_updated_df = pd.concat([kb_df.drop(columns=["char_count", "word_count"]), kb_update_df], ignore_index=True)

updated_chunks_df = build_chunks(kb_updated_df, chunk_size=MAIN_CHUNK_SIZE, overlap=MAIN_OVERLAP)
updated_system = build_retrieval_system(updated_chunks_df)

print("Original document count:", len(kb_df))
print("Updated document count:", len(kb_updated_df))
print("Original chunk count:", len(chunks_df))
print("Updated chunk count:", len(updated_chunks_df))

In [ ]:
update_queries = [
    "Why can overlap help when a fact sits near a chunk boundary?",
    "What does hybrid retrieval combine?",
    "What failure modes are common in mini RAG systems?",
    "How should we debug whether retrieval or answer generation caused an error?",
]

before_after_rows = []
for query in update_queries:
    before_df = retrieve(query, retrieval_system, top_k=TOP_K)
    after_df = retrieve(query, updated_system, top_k=TOP_K)

    before_sources = unique_in_order(before_df["source"].tolist())
    after_sources = unique_in_order(after_df["source"].tolist())

    before_after_rows.append(
        {
            "query": query,
            "before_retrieved_sources": " | ".join(before_sources),
            "after_retrieved_sources": " | ".join(after_sources),
            "changed": bool(before_sources != after_sources),
        }
    )

before_after_df = pd.DataFrame(before_after_rows)
before_after_path = ARTIFACTS_DIR / "retrieval_before_after_update.csv"
before_after_df.to_csv(before_after_path, index=False)
before_after_df

## 7. Mini-RAG

Для учебной версии mini-RAG не используем внешнюю LLM.
Ответ строится из предложений с максимальным пересечением по ключевым словам с вопросом, а рядом всегда возвращаются источники.

In [ ]:
def tokenize_keywords(text: str):
    return [token for token in re.findall(r"[A-Za-z]{4,}", text.lower())]


def build_answer_from_context(question: str, retrieved_df: pd.DataFrame) -> str:
    keywords = set(tokenize_keywords(question))
    sentence_pool = []
    for _, row in retrieved_df.iterrows():
        sentences = re.split(r"(?<=[.!?])\s+", row["chunk_text"])
        for sentence in sentences:
            clean_sentence = sentence.strip()
            if not clean_sentence:
                continue
            sentence_tokens = set(tokenize_keywords(clean_sentence))
            overlap = len(keywords & sentence_tokens)
            score = overlap + (1.0 / row["rank"])
            sentence_pool.append((score, row["source"], clean_sentence))

    sentence_pool.sort(key=lambda item: item[0], reverse=True)

    selected = []
    seen_sentences = set()
    for score, source, sentence in sentence_pool:
        if sentence in seen_sentences:
            continue
        selected.append(sentence)
        seen_sentences.add(sentence)
        if len(selected) == 3:
            break

    if not selected:
        selected = retrieved_df["chunk_text"].head(2).tolist()

    return " ".join(selected)


def answer_question(question: str, system: dict, top_k: int = 3) -> dict:
    retrieved_df = retrieve(question, system, top_k=top_k)
    answer = build_answer_from_context(question, retrieved_df)
    sources = unique_in_order(retrieved_df["source"].tolist())
    return {
        "question": question,
        "answer": answer,
        "retrieved_sources": " | ".join(sources),
        "context_preview": " ".join(retrieved_df["chunk_text"].head(2).tolist())[:280] + "...",
    }

In [ ]:
rag_questions = [
    "Why should a retrieval notebook keep a final test split untouched?",
    "How can TF-IDF vectors be searched with FAISS?",
    "Why is overlap useful for chunking?",
    "What does hybrid retrieval combine?",
    "What usually goes wrong in mini RAG systems?",
]

rag_rows = [answer_question(question, updated_system, top_k=TOP_K) for question in rag_questions]
rag_examples_df = pd.DataFrame(rag_rows)
rag_examples_path = ARTIFACTS_DIR / "rag_examples.csv"
rag_examples_df.to_csv(rag_examples_path, index=False)
rag_examples_df[["question", "answer", "retrieved_sources"]]

## 8. Краткий анализ ошибок и ограничений

Ниже показаны несколько пограничных вопросов.
Они полезны не потому, что всегда проваливаются полностью, а потому что хорошо демонстрируют ограничения текущего pipeline.

In [ ]:
borderline_cases = [
    {
        "question": "Why can retrieval quality look too optimistic when future information is present?",
        "likely_issue": "The query overlaps with both leakage and time-series documents, so retrieval can split evidence across sources.",
        "main_component": "retrieval + context assembly",
    },
    {
        "question": "How should we keep neighboring facts together without creating too many redundant chunks?",
        "likely_issue": "The question mixes chunk_size and overlap, so multiple chunking documents may compete for rank one.",
        "main_component": "chunking design",
    },
    {
        "question": "Why might a template answer sound repetitive even with relevant context?",
        "likely_issue": "Retrieval can be correct, but the answer builder only copies top-scoring sentences instead of rewriting them.",
        "main_component": "answer generation",
    },
]

borderline_df = pd.DataFrame(borderline_cases)
borderline_df

In [ ]:
best_experiment_row = experiment_df.sort_values(["hit_at_k", "recall_at_k", "mrr_at_k"], ascending=False).iloc[0]
simplest_queries = retrieval_eval_df.loc[retrieval_eval_df["rank_of_first_relevant"] == 1, "query"].tolist()
hardest_queries = retrieval_eval_df.sort_values(["hit_at_k", "recall_at_k", "rank_of_first_relevant"], ascending=[True, True, False]).head(3)["query"].tolist()

report_text = f'''# HW14 – эмбеддинги, FAISS, оценка retrieval и mini-RAG по базе знаний

## 1. Кратко: что сделано

- Использована локально собранная база знаний по темам retrieval, ML workflow и mini-RAG.
- Документы были разбиты на фиксированные символьные чанки с overlap.
- В качестве векторизации использовался `TfidfVectorizer`, а поиск выполнялся через `FAISS IndexFlatIP`.
- Базовая проверка retrieval показала `hit@{TOP_K} = {retrieval_metrics["hit_at_k"]:.3f}` и `recall@{TOP_K} = {retrieval_metrics["recall_at_k"]:.3f}` на контрольных запросах.
- После обновления базы знаний retrieval изменился на запросах про overlap, hybrid retrieval и failure modes mini-RAG.
- Mini-RAG формирует ответ из лучших предложений найденного контекста и всегда возвращает источники.

## 2. Среда и воспроизводимость

- Python: {sys.version.split()[0]}
- faiss / faiss-cpu: {getattr(faiss, "__version__", "faiss-cpu")}
- sentence-transformers / transformers / sklearn: sklearn {__import__("sklearn").__version__}
- torch (если использовался): not used
- Устройство (CPU/GPU): CPU
- Seed: {SEED}
- Как запустить: открыть `HW14.ipynb` и выполнить Run All.

## 3. База знаний и подготовка документов

- Предметная область: retrieval, chunking, evaluation, mini-RAG и related ML workflow topics.
- Источник документов: локально собранная учебная база знаний в `knowledge_base.json`.
- Число исходных документов: {len(kb_df)}
- Число чанков после разбиения: {len(chunks_df)}
- Какой способ чанкинга использовался: fixed-size character chunking with overlap.
- Основные параметры чанкинга: `chunk_size={MAIN_CHUNK_SIZE}`, `overlap={MAIN_OVERLAP}`.
- Комментарий (3-6 предложений): база знаний получилась небольшой, но тематически связанной. Внутри документов есть как близкие темы, так и разные аспекты retrieval pipeline, поэтому retrieval не сводится к точному keyword match. Такой корпус хорошо подходит для учебного mini-RAG, потому что на нём можно показать влияние chunking, оценку retrieval и обновление базы знаний без тяжёлой инфраструктуры. Документы достаточно короткие, чтобы ноутбук выполнялся быстро, но после чанкинга остаётся достаточно фрагментов для meaningful top-k поиска.

## 4. Retrieval и индекс

### 4.1. Эмбеддинги и индекс

- Какая embedding-модель или способ векторизации использовались: `TfidfVectorizer(ngram_range=(1, 2))`.
- Какая метрика сходства использовалась: cosine-like similarity through L2 normalization + inner product.
- Какой индекс `FAISS` использовался: `IndexFlatIP`.
- Какой `top_k` использовался по умолчанию: {TOP_K}
- Что показали первые примеры retrieval: на прямых вопросах система обычно возвращала релевантный источник на первой позиции, а на более абстрактных формулировках иногда смешивала соседние по теме документы.

### 4.2. Контрольные запросы и оценка retrieval

- Сколько контрольных запросов использовалось: {len(eval_queries)}
- Что считалось релевантным ответом: попадание хотя бы одного ожидаемого source id в top-k retrieved sources.
- Какие метрики retrieval считались: `hit@k`, `recall@k`, `MRR@k`.
- Итоговый `hit@k`: {retrieval_metrics["hit_at_k"]:.3f}
- Итоговый `recall@k`: {retrieval_metrics["recall_at_k"]:.3f}
- Дополнительные метрики (если были): `MRR@k = {retrieval_metrics["mrr_at_k"]:.3f}`.
- Какие запросы оказались самыми простыми: {", ".join(simplest_queries[:3]) if simplest_queries else "n/a"}
- Какие запросы оказались самыми проблемными: {", ".join(hardest_queries)}

## 5. Эксперимент по параметрам retrieval и обновление базы знаний

Опишите коротко и по делу:

- Какой параметр сравнивался (`chunk_size`, `overlap`, `top_k` или другой разумный вариант): `chunk_size`.
- Какие два варианта сравнивались: `220` и `320`.
- Как изменилось качество retrieval: smaller chunks produced more fragments, while the larger chunk size gave the best balance of context and retrieval quality in this notebook.
- Какой вариант вы выбрали как основной и почему: `chunk_size={int(best_experiment_row["chunk_size"])}` because it achieved the strongest aggregate retrieval metrics.
- Что именно было обновлено в базе знаний: добавлены документы про overlap tuning, hybrid retrieval и типичные failure modes mini-RAG.
- Как была выполнена переиндексация: новые документы были объединены с исходной базой, затем pipeline заново выполнил chunking, vectorization and FAISS indexing.
- Что изменилось в retrieval после обновления базы знаний: на запросах из `retrieval_before_after_update.csv` появились новые целевые источники, а retrieved source lists заметно изменились.

## 6. Mini-RAG и результаты

Ссылки на файлы в репозитории:

- Оценка retrieval: `./artifacts/retrieval_eval.csv`
- Примеры ответов mini-RAG: `./artifacts/rag_examples.csv`
- Сравнение до/после обновления: `./artifacts/retrieval_before_after_update.csv`

Короткая сводка (6-10 строк):

- Как формировался контекст для ответа: брались top-{TOP_K} чанков и из них собирался небольшой sentence pool.
- Какой способ генерации ответа использовался: template-based extractive answer builder.
- Возвращались ли источники вместе с ответом: yes, always.
- На каких вопросах mini-RAG сработал лучше всего: на вопросах с прямой терминологией про splits, FAISS, TF-IDF и source attribution.
- На каких вопросах mini-RAG ошибался или отвечал неуверенно: на более абстрактных вопросах, где ответ распределён между несколькими документами или требуется сильная переформулировка.
- Какую роль сыграло качество retrieval в итоговом ответе: retrieval определяло почти всё качество mini-RAG, потому что генерация была очень простой и не могла компенсировать плохой контекст.

## 7. Анализ

Выбранная база знаний оказалась удачной для учебной задачи, потому что она достаточно компактна для быстрого запуска и при этом содержит несколько близких тем, которые реально конкурируют в retrieval. Чанкинг фиксированными окнами оказался понятным и воспроизводимым baseline. При `chunk_size=320` система чаще сохраняла локальный контекст определения и объяснения в одном фрагменте, поэтому retrieval был стабильнее, чем при меньшем размере чанка. Формальная оценка retrieval оказалась полезной, потому что без неё было бы сложно отличить случайно хорошие примеры от устойчивого поведения системы. Метрики `hit@k`, `recall@k` и `MRR@k` дали достаточно компактную картину качества на наборе контрольных запросов. Эксперимент с параметром `chunk_size` показал ожидаемый tradeoff между точностью фрагмента и полнотой локального контекста. Обновление базы знаний было особенно показательным, потому что на части запросов до апдейта система искала только соседние по смыслу документы, а после апдейта начала находить точные новые источники. В mini-RAG стало хорошо видно, что даже корректный retrieval не гарантирует идеально сформулированный ответ, если генерация остаётся шаблонной. Самые показательные ошибки связаны либо с расплывчатой формулировкой вопроса, либо с тем, что нужная информация распределена по нескольким chunk-ам. Ещё одно ограничение состоит в том, что TF-IDF чувствителен к лексике и хуже переносит сильные перефразировки. Кроме того, extractive answer builder склонен повторять фразы документов вместо более аккуратного синтеза. Для учебного ноутбука это приемлемо, но для более серьёзного RAG понадобились бы richer embeddings, reranking и более сильный answer generator.

## 8. Итоговый вывод

Для такой задачи я бы оставил базовый retrieval-конвейер с fixed-size chunking, умеренным overlap, TF-IDF vectorization и `FAISS IndexFlatIP`, потому что он простой, быстрый и прозрачно диагностируется. Главное наблюдение про качество retrieval состоит в том, что даже маленькие решения нужно оценивать формально, а не только по нескольким удачным примерам. Ещё один важный вывод в том, что mini-RAG напрямую зависит от retrieval: если контекст слабый или неполный, простая генерация почти не умеет это исправлять.

## 9. Приложение (опционально)

- Дополнительный benchmark по `chunk_size`: `./artifacts/figures/chunk_size_experiment.png`
- График качества retrieval: `./artifacts/figures/retrieval_quality_plot.png`
- Примеры чанков: `./artifacts/chunk_examples.csv`
'''

REPORT_PATH.write_text(report_text, encoding="utf-8")
print("Report saved to:", REPORT_PATH)
print("Artifacts:")
print("-", retrieval_eval_path)
print("-", rag_examples_path)
print("-", before_after_path)